# Chapter 13: Interactive Charts with Plotly

*Data Analytics with Python and Jupyter Notebook*

Run each cell in order with **Shift + Enter**. Every chart in this notebook
is interactive: hover over it, drag to zoom, double-click to reset, and click
legend entries to hide or show a series. (The book can only show static
snapshots.)

Datasets: `bean_there_sales.csv` and `ch12_bean_there_orders.csv` (the
receipts sample from Chapter 12).

## 13.1 Setting up

In [1]:
import os
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.width", 80)
pio.templates.default = "plotly_white"

sales = pd.read_csv("../data/bean_there_sales.csv", parse_dates=["date"])
orders = pd.read_csv("../data/ch12_bean_there_orders.csv",
                     parse_dates=["date"])

The next cell only matters for saving static PNG snapshots (the images in
the book). Kaleido, the library behind `fig.write_image`, drives a headless
Chrome. If Chrome is somewhere unusual, point the `BROWSER_PATH` environment
variable at it. On the machine that built the book it lives in
`/opt/pw-browsers/chromium`. If you have no Chrome at all, run
`plotly_get_chrome` once in a terminal.

In [2]:
# Only needed if Kaleido can't find Chrome on its own
chromium = "/opt/pw-browsers/chromium"   # the book's build machine
if os.path.exists(chromium):
    os.environ["BROWSER_PATH"] = chromium

In [3]:
# Folder and size settings for the book's PNG snapshots
from pathlib import Path

FIG = Path("../figures")
FIG.mkdir(exist_ok=True)
SNAP = {"width": 900, "height": 450, "scale": 2}

## 13.2 Your first interactive chart

Hover, drag to zoom, double-click to reset, click legend entries to hide a store.

In [4]:
daily = (
    sales.groupby(["date", "store"], as_index=False)["revenue"]
    .sum()
)
daily.head(3)

,date,store,revenue
0,2026-01-01,Downtown,696.10
1,2026-01-01,Riverside,310.45
2,2026-01-01,University,568.45


In [5]:
fig = px.line(daily, x="date", y="revenue", color="store",
              title="Daily revenue by store, Q1 2026")
fig.show()

In [6]:
fig.write_image(FIG / "ch13-line-daily.png", **SNAP)

### What is inside a Plotly figure?

In [7]:
print(type(fig))
print(len(fig.data), "traces")
for trace in fig.data:
    print(trace.type, trace.name, len(trace.x), "points")
print("Title:", fig.layout.title.text)

<class 'plotly.graph_objs._figure.Figure'>
3 traces
scatter Downtown 90 points
scatter Riverside 90 points
scatter University 90 points
Title: Daily revenue by store, Q1 2026


## 13.3 A tour of Plotly Express

### Bar charts

`px.bar` draws values as given, so aggregate with pandas first. `color_discrete_map` keeps each store's color consistent.

In [8]:
store_order = ["Downtown", "University", "Riverside"]
STORE_COLORS = {"Downtown": "#4C72B0", "University": "#DD8452",
                "Riverside": "#55A868"}

by_product = (
    sales.groupby(["product", "store"], as_index=False)["revenue"]
    .sum()
    .sort_values("revenue", ascending=False)
)
fig = px.bar(by_product, x="product", y="revenue", color="store",
             barmode="group",
             category_orders={"store": store_order},
             color_discrete_map=STORE_COLORS,
             title="Q1 revenue by product and store")
fig.show()

In [9]:
fig.write_image(FIG / "ch13-bar-product.png", **SNAP)

### Scatter plots with rich hover details

`hover_data` adds columns to the tooltip; `labels` renames columns everywhere.

In [10]:
hourly = (
    orders.groupby(["store", "hour"], as_index=False)
    .agg(orders=("order_id", "count"),
         avg_wait=("wait_min", "mean"),
         avg_rating=("rating", "mean"))
)
hourly.round(2).head(3)

,store,hour,orders,avg_wait,avg_rating
0,Downtown,7,92,5.67,3.85
1,Downtown,8,198,6.76,3.40
2,Downtown,9,202,6.39,3.65


In [11]:
fig = px.scatter(
    hourly, x="orders", y="avg_wait", color="store",
    category_orders={"store": store_order},
    color_discrete_map=STORE_COLORS,
    hover_data={"hour": True, "avg_rating": ":.2f",
                "avg_wait": ":.1f"},
    labels={"orders": "Orders in sample", "avg_wait": "Avg wait (min)",
            "avg_rating": "Avg rating", "hour": "Hour"},
    title="Busier hours mean longer waits",
)
fig.update_traces(marker_size=11)
fig.show()

### Distributions: box plots

In [12]:
fig = px.box(orders, x="store", y="wait_min", color="store",
             category_orders={"store": store_order},
             color_discrete_map=STORE_COLORS,
             labels={"wait_min": "Wait (minutes)", "store": ""},
             title="Wait times by store")
fig.update_layout(showlegend=False)
fig.show()

In [13]:
fig.write_image(FIG / "ch13-box-wait.png", **SNAP)

## 13.4 Customizing hover, labels and layout

Try the range slider under the chart and the 1w / 1m / All buttons.

In [14]:
fig = px.line(daily, x="date", y="revenue", color="store",
              category_orders={"store": store_order},
              color_discrete_map=STORE_COLORS,
              labels={"date": "", "revenue": "Revenue ($)",
                      "store": "Store"},
              title="Daily revenue by store, Q1 2026")
fig.update_traces(hovertemplate="%{y:$,.0f}")
fig.update_layout(hovermode="x unified",
                  legend=dict(orientation="h", y=1.02, x=1,
                              xanchor="right", yanchor="bottom"))
fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(buttons=[
        dict(count=7, label="1w", step="day", stepmode="backward"),
        dict(count=1, label="1m", step="month", stepmode="backward"),
        dict(step="all", label="All"),
    ]),
)
fig.show()

## 13.5 A simple dashboard

### Several charts in one figure

Only full Monday-to-Sunday weeks are kept, so the first and last points are not partial weeks.

In [15]:
full_weeks = sales[sales["date"].between("2026-01-05", "2026-03-29")]
weekly = (
    full_weeks.set_index("date")
    .groupby("store")["revenue"]
    .resample("W")
    .sum()
    .reset_index()
)
weekly.head(3)

,store,date,revenue
0,Downtown,2026-01-11,4953.10
1,Downtown,2026-01-18,4926.25
2,Downtown,2026-01-25,5306.25


In [16]:
product_rev = sales.groupby("product")["revenue"].sum().sort_values()
rating = orders.groupby("store")["rating"].mean().reindex(store_order)

board = make_subplots(
    rows=2, cols=2,
    subplot_titles=["Weekly revenue ($)", "Q1 revenue by product ($)",
                    "Wait time (minutes)", "Average rating (1 to 5)"],
    vertical_spacing=0.15, horizontal_spacing=0.14,
)

for store in store_order:
    color = STORE_COLORS[store]
    one = weekly[weekly["store"] == store]
    board.add_trace(
        go.Scatter(x=one["date"], y=one["revenue"], name=store,
                   mode="lines+markers", legendgroup=store,
                   line_color=color),
        row=1, col=1,
    )
    waits = orders.loc[orders["store"] == store, "wait_min"]
    board.add_trace(
        go.Box(y=waits, name=store, legendgroup=store,
               showlegend=False, marker_color=color),
        row=2, col=1,
    )

board.add_trace(
    go.Bar(x=product_rev.values, y=product_rev.index, orientation="h",
           marker_color="#4C72B0", showlegend=False),
    row=1, col=2,
)
board.add_trace(
    go.Bar(x=rating.index, y=rating.values, showlegend=False,
           marker_color=[STORE_COLORS[s] for s in rating.index],
           texttemplate="%{y:.2f}", textposition="outside"),
    row=2, col=2,
)

board.update_yaxes(range=[0, 5], row=2, col=2)
board.update_layout(title="Bean There, Q1 2026 at a glance",
                    height=650, width=1000)
board.show()

### A dropdown to pick the store

All three stores' bars are in the figure; each button changes which one is visible.

In [17]:
pivot = (
    sales.pivot_table(index="product", columns="store",
                      values="revenue", aggfunc="sum")
    .sort_values("Downtown", ascending=False)
)

fig = go.Figure()
for store in store_order:
    fig.add_trace(go.Bar(x=pivot.index, y=pivot[store], name=store,
                         marker_color=STORE_COLORS[store],
                         visible=(store == "Downtown")))

buttons = []
for i, store in enumerate(store_order):
    visible = [j == i for j in range(len(store_order))]
    buttons.append(dict(label=store, method="update",
                        args=[{"visible": visible},
                              {"title.text": f"Q1 revenue: {store}"}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, x=1, xanchor="right", y=1.15)],
    title="Q1 revenue: Downtown",
    yaxis_title="Revenue ($)",
)
fig.show()

In [18]:
fig.write_image(FIG / "ch13-dropdown.png", **SNAP)

## 13.6 Saving and sharing

### Static images

In [19]:
board.write_image("../figures/ch13-dashboard.png", width=1000,
                  height=650, scale=2)

### Standalone HTML

The default file embeds plotly.js (works offline); `include_plotlyjs="cdn"` links to it online (tiny file).

In [20]:
board.write_html("ch13_dashboard.html")
board.write_html("ch13_dashboard_cdn.html", include_plotlyjs="cdn")

for name in ["ch13_dashboard.html", "ch13_dashboard_cdn.html"]:
    size_kb = os.path.getsize(name) / 1024
    print(f"{name:<26}{size_kb:>8,.0f} KB")

ch13_dashboard.html          4,745 KB
ch13_dashboard_cdn.html         43 KB


## Exercises

Solutions are in Appendix D of the book.

1. Draw an interactive histogram of `order_total` with `px.histogram`,
   coloured by `customer`, with `barmode="overlay"` and `opacity=0.6`.
2. Turn the daily revenue line chart into three small charts, one per
   store, with `facet_col="store"`. Then add `facet_col_wrap=1`.
3. Make a scatter plot of the receipts (`wait_min` against `order_total`)
   where hovering shows the `order_id`, `date` and `rating`. *Hint:*
   `hover_name` and `hover_data`.
4. Change the hover text of the bar chart in 13.3 so it reads like
   "Latte at Downtown: $21,029". *Hint:* `customdata` holds extra columns,
   or use `fig.update_traces(hovertemplate=...)` with `%{x}` and `%{y}`.
5. Save the 13.3 bar chart as a standalone HTML file using
   `include_plotlyjs="cdn"`, open it in your browser, and check how big the
   file is.
6. Add a fourth button, "All stores", to the dropdown chart that shows all
   three stores side by side.
7. **Think about it:** For each chart in this notebook, decide whether
   interactivity adds anything a static chart could not. Which would you
   send to the owner as a PNG, and which as HTML?